In [ ]:
import os
import shutil
import kagglehub

# Download competition data
path = kagglehub.competition_download('playground-series-s6e10')
print("Path to competition files:", path)

# Auto-download original dataset for og_* models (safe fallback if offline)
EXTRA_DATA_PATH = None
try:
    orig_dir = kagglehub.dataset_download('teejmahal20/airline-passenger-satisfaction')
    cand_files = [os.path.join(orig_dir, f) for f in os.listdir(orig_dir) if f.endswith('.csv')]
    if cand_files:
        EXTRA_DATA_PATH = cand_files
        print("Found original dataset files:", EXTRA_DATA_PATH)
except Exception as e:
    print("Original dataset not available offline, continuing with competition data only:", e)

In [ ]:
import pandas as pd

train_path = os.path.join(path, "train.csv")
test_path = os.path.join(path, "test.csv")

train_pd = pd.read_csv(train_path)
test_pd = pd.read_csv(test_path)

print("train:", train_pd.shape, "| test:", test_pd.shape)

In [ ]:
import numpy as np

raw_features = [c for c in train_pd.columns if c not in ('id', 'satisfaction')]
encoded_cols = ['Gender', 'Customer Type', 'Type of Travel', 'Class']

rating_cols = [
    c for c in raw_features
    if c not in encoded_cols
    and train_pd[c].min() >= 0 and train_pd[c].max() <= 5
    and train_pd[c].nunique() <= 6
]

dep_col = next(c for c in raw_features if 'departure delay' in c.lower())
arr_col = next(c for c in raw_features if 'arrival delay' in c.lower())

def _keep(cols):
    return [c for c in cols if c in train_pd.columns]

SURVEY_COLS = _keep([
    'Inflight wifi service', 'Departure/Arrival time convenient',
    'Ease of Online booking', 'Gate location', 'Food and drink',
    'Online boarding', 'Seat comfort', 'Inflight entertainment',
    'On-board service', 'Leg room service', 'Baggage handling',
    'Checkin service', 'Cleanliness', 'Inflight service'
])
COMFORT_COLS = _keep(['Seat comfort', 'Leg room service', 'Food and drink', 'Cleanliness', 'Inflight entertainment'])
GROUND_COLS  = _keep(['Departure/Arrival time convenient', 'Gate location', 'Baggage handling', 'Checkin service'])
DIGITAL_COLS = _keep(['Ease of Online booking', 'Online boarding', 'Inflight wifi service'])
ONBOARD_COLS = _keep(['On-board service', 'Inflight service', 'Baggage handling', 'Cleanliness'])

combined_raw = pd.concat([train_pd[raw_features], test_pd[raw_features]], ignore_index=True)

count_cols = ['Age', 'Flight Distance', dep_col, arr_col]
count_maps = {c: combined_raw[c].fillna(-1).value_counts() for c in count_cols}

# Joint frequency maps for synthetic generator artifacts
dist_class_vc = (combined_raw['Flight Distance'].astype(str) + '_' + combined_raw['Class'].astype(str)).value_counts()
age_travel_vc = (combined_raw['Age'].astype(str) + '_' + combined_raw['Type of Travel'].astype(str)).value_counts()
delay_pair_vc = (combined_raw[dep_col].fillna(-1).astype(int).astype(str) + '_' + combined_raw[arr_col].fillna(-1).astype(int).astype(str)).value_counts()

def encode_categoricals(df):
    """Map categorical string columns to numeric values with proper ordinal order for Class."""
    df = df.copy()
    df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0}).fillna(0).astype(np.int8)
    df['Customer Type'] = df['Customer Type'].map({'Loyal Customer': 1, 'disloyal Customer': 0}).fillna(0).astype(np.int8)
    df['Type of Travel'] = df['Type of Travel'].map({'Personal Travel': 1, 'Business travel': 0}).fillna(0).astype(np.int8)
    df['Class'] = df['Class'].map({'Eco': 0, 'Eco Plus': 1, 'Business': 2}).fillna(0).astype(np.int8)
    return df

_comb_enc = encode_categoricals(combined_raw)
_comb_enc['Service_Mean_tmp'] = _comb_enc[SURVEY_COLS].mean(axis=1)
_comb_enc['Age_Bin_tmp'] = pd.cut(_comb_enc['Age'], bins=[-1, 18, 26, 38, 52, 65, 120], labels=False).astype(np.int8)

cohort_keys = ['Class', 'Type of Travel', 'Customer Type']
cohort_stats = _comb_enc.groupby(cohort_keys).agg(
    cohort_dist_mean=('Flight Distance', 'mean'),
    cohort_dist_std=('Flight Distance', 'std'),
    cohort_srv_mean=('Service_Mean_tmp', 'mean'),
    cohort_srv_std=('Service_Mean_tmp', 'std'),
    cohort_wifi_mean=('Inflight wifi service', 'mean'),
    cohort_board_mean=('Online boarding', 'mean'),
    cohort_ent_mean=('Inflight entertainment', 'mean'),
    cohort_seat_mean=('Seat comfort', 'mean'),
).reset_index()

age_cohort_keys = ['Age_Bin_tmp', 'Class']
age_cohort_stats = _comb_enc.groupby(age_cohort_keys).agg(
    age_cohort_srv_mean=('Service_Mean_tmp', 'mean'),
    age_cohort_wifi_mean=('Inflight wifi service', 'mean'),
    age_cohort_board_mean=('Online boarding', 'mean'),
).reset_index()

def add_features(df, is_orig=0):
    """Add domain-aware engineered features on top of already-encoded data."""
    df = df.copy()
    orig_index = df.index
    for c in encoded_cols:
        df[c] = pd.to_numeric(df[c])

    # Domain indicator (0 = competition train/test, 1 = external original dataset)
    df['is_original'] = np.int8(is_orig)

    # 1. Frequency & synthetic artifact encodings
    for c in count_cols:
        df[f'{c}_count'] = df[c].fillna(-1).map(count_maps[c]).fillna(1).astype(np.float32)

    raw_class_str = df['Class'].map({0: 'Eco', 1: 'Eco Plus', 2: 'Business'})
    raw_travel_str = df['Type of Travel'].map({1: 'Personal Travel', 0: 'Business travel'})
    df['Dist_Class_count'] = (df['Flight Distance'].astype(str) + '_' + raw_class_str.astype(str)).map(dist_class_vc).fillna(1).astype(np.float32)
    df['Age_Travel_count'] = (df['Age'].astype(str) + '_' + raw_travel_str.astype(str)).map(age_travel_vc).fillna(1).astype(np.float32)
    df['Delay_Pair_count'] = (df[dep_col].fillna(-1).astype(int).astype(str) + '_' + df[arr_col].fillna(-1).astype(int).astype(str)).map(delay_pair_vc).fillna(1).astype(np.float32)

    # 2. Distance & Age transformations
    df['Distance_Log']    = np.log1p(df['Flight Distance']).astype(np.float32)
    df['Distance_Sqrt']   = np.sqrt(df['Flight Distance']).astype(np.float32)
    df['Dist_mod_10']     = (df['Flight Distance'] % 10).astype(np.int8)
    df['Dist_mod_100']    = (df['Flight Distance'] % 100).astype(np.int16)
    df['Is_Long_Haul']    = (df['Flight Distance'] > 2000).astype(np.int8)
    df['Is_Short_Haul']   = (df['Flight Distance'] < 500).astype(np.int8)
    df['Age_Bin']         = pd.cut(df['Age'], bins=[-1, 18, 26, 38, 52, 65, 120], labels=False).astype(np.int8)
    df['Is_Young_Adult']  = ((df['Age'] >= 22) & (df['Age'] <= 60)).astype(np.int8)
    df['Age_x_Class']     = (df['Age'] * (df['Class'] + 1)).astype(np.int16)

    # 3. Delay features
    df['Arrival_Delay_Missing'] = df[arr_col].isna().astype(np.int8)
    arr_delay = df[arr_col].fillna(df[dep_col].fillna(0))
    dep_delay = df[dep_col].fillna(0)
    df[arr_col] = arr_delay
    df['Total_Delay']       = (dep_delay + arr_delay).astype(np.float32)
    df['Delay_Delta']       = (arr_delay - dep_delay).astype(np.float32)
    df['Delay_Ratio']       = (arr_delay / (dep_delay + 1.0)).astype(np.float32)
    df['Delay_Log_Total']   = np.log1p(np.maximum(0, df['Total_Delay'])).astype(np.float32)
    df['Dep_Delay_Log']     = np.log1p(np.maximum(0, dep_delay)).astype(np.float32)
    df['Arr_Delay_Log']     = np.log1p(np.maximum(0, arr_delay)).astype(np.float32)
    df['Delay_Per_Mile']    = (df['Total_Delay'] / (df['Flight Distance'] + 10.0)).astype(np.float32)
    df['Is_Delayed']        = ((dep_delay > 0) | (arr_delay > 0)).astype(np.int8)
    df['Is_Arr_Delayed_15'] = (arr_delay >= 15).astype(np.int8)
    df['Recovered_In_Air']  = ((dep_delay > 0) & (arr_delay == 0)).astype(np.int8)
    df['Severe_Delay_Flag'] = ((dep_delay > 45) | (arr_delay > 45)).astype(np.int8)

    # 4. Sub-index aggregations
    df['Inflight_Comfort_Index'] = df[COMFORT_COLS].mean(axis=1).astype(np.float32)
    df['Ground_Service_Index']   = df[GROUND_COLS].mean(axis=1).astype(np.float32)
    df['Digital_Boarding_Index'] = df[DIGITAL_COLS].mean(axis=1).astype(np.float32)
    df['Onboard_Crew_Index']     = df[ONBOARD_COLS].mean(axis=1).astype(np.float32)
    df['Comfort_vs_Ground_Diff'] = (df['Inflight_Comfort_Index'] - df['Ground_Service_Index']).astype(np.float32)
    df['Digital_vs_Ground_Diff'] = (df['Digital_Boarding_Index'] - df['Ground_Service_Index']).astype(np.float32)
    df['Digital_vs_Comfort_Diff']= (df['Digital_Boarding_Index'] - df['Inflight_Comfort_Index']).astype(np.float32)

    # 5. Full & Non-Zero (NZ) Survey Rating Profiles
    survey_vals = df[SURVEY_COLS].values
    df['Service_Mean']  = survey_vals.mean(axis=1).astype(np.float32)
    df['Service_Std']   = survey_vals.std(axis=1).astype(np.float32)
    df['Service_Min']   = survey_vals.min(axis=1).astype(np.int8)
    df['Service_Max']   = survey_vals.max(axis=1).astype(np.int8)
    df['Service_Range'] = (df['Service_Max'] - df['Service_Min']).astype(np.int8)
    df['Service_Skew']  = df[SURVEY_COLS].skew(axis=1).fillna(0).astype(np.float32)

    # Separate 0 ("Not Applicable") from true ratings (1-5)
    nz_mask = survey_vals > 0
    nz_count = np.maximum(1, nz_mask.sum(axis=1))
    nz_sum = np.where(nz_mask, survey_vals, 0).sum(axis=1)
    nz_mean = (nz_sum / nz_count).astype(np.float32)
    nz_var = np.where(nz_mask, (survey_vals - nz_mean[:, None]) ** 2, 0).sum(axis=1) / nz_count
    nz_min = np.where(nz_mask, survey_vals, 99).min(axis=1)
    nz_min = np.where(nz_min == 99, 0, nz_min).astype(np.int8)

    df['Service_Mean_NZ']  = nz_mean
    df['Service_Std_NZ']   = np.sqrt(nz_var).astype(np.float32)
    df['Service_Min_NZ']   = nz_min
    df['Service_Range_NZ'] = (df['Service_Max'] - df['Service_Min_NZ']).astype(np.int8)

    for r in range(6):
        df[f'Rating_{r}_Count'] = (survey_vals == r).sum(axis=1).astype(np.int8)
    df['Zero_Rating_Count'] = df['Rating_0_Count']
    df['High_Rating_Count'] = (survey_vals >= 4).sum(axis=1).astype(np.int8)
    df['Low_Rating_Count']  = ((survey_vals >= 1) & (survey_vals <= 2)).sum(axis=1).astype(np.int8)
    df['Mid_Rating_Count']  = (survey_vals == 3).sum(axis=1).astype(np.int8)
    df['High_to_Low_Diff']  = (df['High_Rating_Count'] - df['Low_Rating_Count']).astype(np.int8)
    df['High_Rating_Ratio_NZ'] = (df['High_Rating_Count'] / nz_count).astype(np.float32)
    df['Low_Rating_Ratio_NZ']  = (df['Low_Rating_Count'] / nz_count).astype(np.float32)
    df['All_Satisfied_NZ']     = ((df['Low_Rating_Count'] == 0) & (df['Mid_Rating_Count'] == 0) & (df['High_Rating_Count'] > 0)).astype(np.int8)

    # Explicit N/A (0) indicators for key digital/gate questions
    df['Wifi_Is_Zero']     = (df['Inflight wifi service'] == 0).astype(np.int8)
    df['Boarding_Is_Zero'] = (df['Online boarding'] == 0).astype(np.int8)
    df['Booking_Is_Zero']  = (df['Ease of Online booking'] == 0).astype(np.int8)

    # Weighted domain score based on top SHAP importances
    df['Weighted_Key_Score'] = (
        0.30 * df['Online boarding']
        + 0.25 * df['Inflight wifi service']
        + 0.15 * df['Inflight entertainment']
        + 0.10 * df['Seat comfort']
        + 0.10 * df['Leg room service']
        + 0.10 * df['On-board service']
    ).astype(np.float32)

    # 6. Pairwise & 3-way Survey Interactions
    if {'Inflight wifi service', 'Ease of Online booking'} <= set(df.columns):
        df['Wifi_Booking_Diff'] = (df['Inflight wifi service'] - df['Ease of Online booking']).astype(np.int8)
        df['Wifi_Booking_Prod'] = (df['Inflight wifi service'] * df['Ease of Online booking']).astype(np.int8)
    if {'Seat comfort', 'Leg room service'} <= set(df.columns):
        df['Space_Comfort_Sum']  = (df['Seat comfort'] + df['Leg room service']).astype(np.int8)
        df['Space_Comfort_Prod'] = (df['Seat comfort'] * df['Leg room service']).astype(np.int8)
    if {'Online boarding', 'Inflight entertainment'} <= set(df.columns):
        df['Boarding_Entertain_Prod'] = (df['Online boarding'] * df['Inflight entertainment']).astype(np.int8)
        df['Boarding_Entertain_Code'] = (df['Online boarding'] * 6 + df['Inflight entertainment']).astype(np.int16)
    if {'Online boarding', 'Inflight wifi service'} <= set(df.columns):
        df['Boarding_Wifi_Prod'] = (df['Online boarding'] * df['Inflight wifi service']).astype(np.int8)
        df['Boarding_Wifi_Diff'] = (df['Online boarding'] - df['Inflight wifi service']).astype(np.int8)
        df['Boarding_Wifi_Code'] = (df['Online boarding'] * 6 + df['Inflight wifi service']).astype(np.int16)
    if {'Seat comfort', 'Cleanliness'} <= set(df.columns):
        df['Seat_Clean_Prod'] = (df['Seat comfort'] * df['Cleanliness']).astype(np.int8)
    if {'Inflight service', 'Baggage handling'} <= set(df.columns):
        df['Inflight_Baggage_Prod'] = (df['Inflight service'] * df['Baggage handling']).astype(np.int8)

    # 7. High-order Categorical Interaction Codes
    df['class_travel']          = (df['Class'] * 2 + df['Type of Travel']).astype(np.int8)
    df['cust_travel']           = (df['Customer Type'] * 2 + df['Type of Travel']).astype(np.int8)
    df['class_cust_travel']     = (df['Class'] * 4 + df['Customer Type'] * 2 + df['Type of Travel']).astype(np.int8)
    df['class_boarding']        = (df['Class'] * 6 + df['Online boarding']).astype(np.int16)
    df['travel_wifi']           = (df['Type of Travel'] * 6 + df['Inflight wifi service']).astype(np.int16)
    df['travel_boarding']       = (df['Type of Travel'] * 6 + df['Online boarding']).astype(np.int16)
    df['cust_wifi']             = (df['Customer Type'] * 6 + df['Inflight wifi service']).astype(np.int16)
    df['class_travel_boarding'] = (df['class_travel'] * 6 + df['Online boarding']).astype(np.int16)
    df['class_travel_wifi']     = (df['class_travel'] * 6 + df['Inflight wifi service']).astype(np.int16)
    df['digital_triad_code']    = (df['Online boarding'] * 36 + df['Inflight wifi service'] * 6 + df['Ease of Online booking']).astype(np.int16)

    # 8. Cohort relative features
    merged = df[cohort_keys].merge(cohort_stats, on=cohort_keys, how='left')
    df['Dist_Diff_Cohort']  = (df['Flight Distance'].values - merged['cohort_dist_mean'].values).astype(np.float32)
    df['Dist_Z_Cohort']     = (df['Dist_Diff_Cohort'].values / (merged['cohort_dist_std'].values + 1.0)).astype(np.float32)
    df['Srv_Diff_Cohort']   = (df['Service_Mean'].values - merged['cohort_srv_mean'].values).astype(np.float32)
    df['Srv_Z_Cohort']      = (df['Srv_Diff_Cohort'].values / (merged['cohort_srv_std'].values + 0.01)).astype(np.float32)
    df['Wifi_Diff_Cohort']  = (df['Inflight wifi service'].values - merged['cohort_wifi_mean'].values).astype(np.float32)
    df['Board_Diff_Cohort'] = (df['Online boarding'].values - merged['cohort_board_mean'].values).astype(np.float32)
    df['Ent_Diff_Cohort']   = (df['Inflight entertainment'].values - merged['cohort_ent_mean'].values).astype(np.float32)
    df['Seat_Diff_Cohort']  = (df['Seat comfort'].values - merged['cohort_seat_mean'].values).astype(np.float32)

    df['Age_Bin_tmp'] = df['Age_Bin']
    merged_age = df[age_cohort_keys].merge(age_cohort_stats, on=age_cohort_keys, how='left')
    df['Srv_Diff_AgeCohort']   = (df['Service_Mean'].values - merged_age['age_cohort_srv_mean'].values).astype(np.float32)
    df['Wifi_Diff_AgeCohort']  = (df['Inflight wifi service'].values - merged_age['age_cohort_wifi_mean'].values).astype(np.float32)
    df['Board_Diff_AgeCohort'] = (df['Online boarding'].values - merged_age['age_cohort_board_mean'].values).astype(np.float32)
    df.drop(columns=['Age_Bin_tmp'], inplace=True)

    df.index = orig_index
    return df

# Apply base feature engineering to train and test
train_pd['satisfaction'] = train_pd['satisfaction'].map(
    lambda v: int(str(v).strip().lower() in ('satisfied', 'true', '1'))
)
train_pd = add_features(encode_categoricals(train_pd), is_orig=0)
test_fe  = add_features(encode_categoricals(test_pd), is_orig=0)

base_features = [c for c in train_pd.columns if c not in ('id', 'satisfaction')]
print(f"{len(raw_features)} raw -> {len(base_features)} base engineered features")

In [ ]:
from sklearn.model_selection import StratifiedKFold

N_FOLDS = 10
y = train_pd['satisfaction'].astype(int)
y_arr = y.values
folds = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42).split(train_pd, y))

TE_SINGLE_COLS = [
    'Flight Distance', 'Age', dep_col, arr_col,
    'Boarding_Wifi_Code', 'class_boarding', 'travel_wifi',
    'class_travel_boarding', 'class_travel_wifi', 'digital_triad_code'
]
TE_PAIR_SPECS = [
    ('Flight Distance', 'Class'),
    ('Flight Distance', 'Type of Travel'),
    ('Flight Distance', 'Online boarding'),
    ('Age', 'Type of Travel'),
    ('Age', 'Class'),
    ('Online boarding', 'Type of Travel'),
    ('Inflight wifi service', 'Type of Travel'),
    ('class_cust_travel', 'Online boarding'),
    ('class_cust_travel', 'Inflight wifi service'),
    ('Seat comfort', 'Class'),
    ('Inflight entertainment', 'Class'),
    (dep_col, arr_col),
]
TE_TRIPLE_SPECS = [
    ('Class', 'Type of Travel', 'Online boarding'),
    ('Class', 'Customer Type', 'Inflight wifi service'),
    ('Online boarding', 'Inflight wifi service', 'Ease of Online booking'),
    ('Class', 'Online boarding', 'Inflight entertainment'),
]

ALL_TE_SPECS = [(c,) for c in TE_SINGLE_COLS] + list(TE_PAIR_SPECS) + list(TE_TRIPLE_SPECS)

def _build_key_series(df, spec):
    s = df[spec[0]].astype(str)
    for col in spec[1:]:
        s = s + '_' + df[col].astype(str)
    return s

def compute_oof_target_encoding(tr_df, te_df, y_ser, folds_list, m_smooth=20.0):
    """Compute leak-free OOF smoothed target encoding (mean, fine-mean, std, WoE) for 1/2/3-way specs."""
    global_mean = float(y_ser.mean())
    global_std  = float(y_ser.std())
    eps_woe = 1e-4
    global_woe = float(np.log( np.clip(global_mean, eps_woe, 1 - eps_woe) / np.clip(1 - global_mean, eps_woe, 1 - eps_woe) ))

    tr_cols, te_cols = {}, {}

    for spec in ALL_TE_SPECS:
        base_name = 'TE_' + '_x_'.join(spec)
        tr_key = _build_key_series(tr_df, spec)
        te_key = _build_key_series(te_df, spec)

        oof_mean = np.full(len(tr_df), global_mean, dtype=np.float32)
        oof_m5   = np.full(len(tr_df), global_mean, dtype=np.float32)
        oof_std  = np.full(len(tr_df), global_std,  dtype=np.float32)

        for tr_idx, va_idx in folds_list:
            k_tr = tr_key.iloc[tr_idx]
            y_tr = y_ser.iloc[tr_idx]
            stats = y_tr.groupby(k_tr).agg(['count', 'mean', 'std'])
            cnt = stats['count']
            mn  = stats['mean']
            sd  = stats['std'].fillna(global_std)

            smooth_20 = (cnt * mn + m_smooth * global_mean) / (cnt + m_smooth)
            smooth_5  = (cnt * mn + 5.0 * global_mean) / (cnt + 5.0)
            smooth_sd = (cnt * sd + 10.0 * global_std) / (cnt + 10.0)

            va_k = tr_key.iloc[va_idx]
            oof_mean[va_idx] = va_k.map(smooth_20).fillna(global_mean).values
            oof_m5[va_idx]   = va_k.map(smooth_5).fillna(global_mean).values
            oof_std[va_idx]  = va_k.map(smooth_sd).fillna(global_std).values

        # Full train mapping for test set
        full_stats = y_ser.groupby(tr_key).agg(['count', 'mean', 'std'])
        f_cnt = full_stats['count']
        f_mn  = full_stats['mean']
        f_sd  = full_stats['std'].fillna(global_std)

        f_smooth_20 = (f_cnt * f_mn + m_smooth * global_mean) / (f_cnt + m_smooth)
        f_smooth_5  = (f_cnt * f_mn + 5.0 * global_mean) / (f_cnt + 5.0)
        f_smooth_sd = (f_cnt * f_sd + 10.0 * global_std) / (f_cnt + 10.0)

        te_mean = te_key.map(f_smooth_20).fillna(global_mean).astype(np.float32).values
        te_m5   = te_key.map(f_smooth_5).fillna(global_mean).astype(np.float32).values
        te_sd   = te_key.map(f_smooth_sd).fillna(global_std).astype(np.float32).values

        tr_cols[base_name] = oof_mean
        te_cols[base_name] = te_mean

        # Add fine-grained smoothing & std & WoE for multi-way and top high-cardinality features
        if len(spec) >= 2 or spec[0] in ('Flight Distance', 'Age', 'digital_triad_code'):
            tr_cols[base_name + '_m5']  = oof_m5
            te_cols[base_name + '_m5']  = te_m5
            tr_cols[base_name + '_std'] = oof_std
            te_cols[base_name + '_std'] = te_sd
            p_tr = np.clip(oof_mean, eps_woe, 1.0 - eps_woe)
            p_te = np.clip(te_mean,  eps_woe, 1.0 - eps_woe)
            tr_cols[base_name + '_woe'] = (np.log(p_tr / (1.0 - p_tr)) - global_woe).astype(np.float32)
            te_cols[base_name + '_woe'] = (np.log(p_te / (1.0 - p_te)) - global_woe).astype(np.float32)

    return pd.DataFrame(tr_cols, index=tr_df.index), pd.DataFrame(te_cols, index=te_df.index)

tr_te_feats, test_te_feats = compute_oof_target_encoding(train_pd, test_fe, y, folds, m_smooth=20.0)
te_col_names = list(tr_te_feats.columns)

train_pd = pd.concat([train_pd, tr_te_feats], axis=1)
test_fe  = pd.concat([test_fe, test_te_feats], axis=1)

features = [c for c in train_pd.columns if c not in ('id', 'satisfaction')]
print(f"Total features with OOF Target Encoding: {len(features)} (added {len(te_col_names)} TE features)")

In [ ]:
!pip install --quiet xgboost optuna catboost pytabkit

## 3. Model Training Setup & Leak-Free Extra Dataset Handler

Key configuration improvements:
- **Leak-Free `X_extra` OOF Encoding:** When `X_extra` is loaded, we compute its target-encoded features via 5-fold cross-validation **within `X_extra` itself** (using competition train `tr` only from the current fold or isolated `X_extra` OOF), completely eliminating validation fold leakage!
- **Down-weighted External Rows (`EXTRA_WEIGHT = 0.50`):** Prevents the external dataset from pulling split thresholds away from the competition's synthetic distribution, directly fixing the `og_lgb` and `og_cat` performance drop seen in v2.
- **Submission Mode Control (`SUBMISSION_TYPE`):** Set `'proba'` (default for ROC AUC evaluation), `'calibrated_proba'` (shifts logits so the optimal accuracy threshold aligns with `0.50`), or `'binary'` (outputs hard `0`/`1` labels if evaluating strict accuracy).

In [ ]:
import os
import re
import shutil
import hashlib
import joblib
import numpy as np
import pandas as pd
import xgboost
import lightgbm as lgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, log_loss)
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from pytabkit import RealMLP_TD_Classifier

HAS_GPU  = bool(shutil.which('nvidia-smi'))
DEVICE   = 'cuda:0' if HAS_GPU else 'cpu'
SCORE_FN = accuracy_score

# Models included in the ensemble:
# - 'xgb', 'lgb', 'cat': Core GBDT triad on competition data
# - 'og_lgb', 'og_cat': Domain-adapted & sample-weighted GBDTs with external dataset
# - 'resnet': Fast GPU PyTorch Tabular ResNet with entity embeddings (~30s/fold)
# - 'mlp': RealMLP_TD_Classifier (pinned to single GPU so it doesn't hang)
MODELS_ON = ['xgb', 'lgb', 'cat', 'og_lgb', 'og_cat', 'resnet', 'mlp']

# Choose submission output format:
# - 'proba'            : calibrated ensemble probabilities in [0, 1] (best for ROC AUC)
# - 'calibrated_proba' : probabilities shifted so optimal accuracy threshold maps to 0.50
# - 'binary'           : hard 0 / 1 integer predictions at the optimal accuracy threshold
SUBMISSION_TYPE = 'proba'

TRAIN_XGB   = True
RESUME      = False
XGB_LR, XGB_NEW_TREES, XGB_PATIENCE = 0.003, 60000, 2000
XGB_SCRATCH = (0.005, 100000, 1000)

LGB_LR, LGB_TREES, LGB_PATIENCE = 0.005, 100000, 1000
CAT_LR, CAT_TREES, CAT_PATIENCE = 0.04, 100000, 1000
MLP_EPOCHS, MLP_BATCH = 25, 2048
RESNET_EPOCHS, RESNET_BATCH = 22, 2048
STACK_ROUNDS, STACK_PATIENCE = 5000, 250
EXTRA_WEIGHT = 0.50 

USE_CKPT, CKPT_DIR = True, 'ckpt_v3'
os.makedirs(CKPT_DIR, exist_ok=True)

xgb_params = {
    'max_depth': 8,
    'min_child_weight': 10,
    'subsample': 0.82,
    'colsample_bytree': 0.55,
    'colsample_bylevel': 0.75,
    'gamma': 0.50,
    'reg_lambda': 2.5,
    'reg_alpha': 5.0,
    'scale_pos_weight': 1.0,
    'max_bin': 4095,
}

lgb_params = dict(
    num_leaves=111,
    max_depth=10,
    min_child_samples=55,
    subsample=0.80,
    subsample_freq=1,
    colsample_bytree=0.52,
    reg_lambda=3.5,
    reg_alpha=1.5,
    path_smooth=5.0,
    max_bin=4095,
)

cat_params = dict(depth=7, l2_leaf_reg=5.0, eval_metric='AUC')

cat_feature_names = (
    encoded_cols + rating_cols
    + ['Age', 'Flight Distance', 'class_travel', 'cust_travel', 'class_cust_travel',
       'Boarding_Wifi_Code', 'class_boarding', 'travel_wifi', 'travel_boarding',
       'cust_wifi', 'class_travel_boarding', 'class_travel_wifi', 'digital_triad_code']
)
cat_feature_names = [c for c in cat_feature_names if c in features]

mlp_cat_cols = [
    c for c in (encoded_cols + rating_cols + [
        'class_travel', 'cust_travel', 'class_cust_travel', 'Age_Bin',
        'Boarding_Wifi_Code', 'class_boarding', 'travel_wifi'
    ])
    if c in features
]

X = train_pd[features].copy()
X_test = test_fe[features].copy()

X[cat_feature_names] = X[cat_feature_names].astype('int64')
X_test[cat_feature_names] = X_test[cat_feature_names].astype('int64')

cat_params.update(cat_features=cat_feature_names, max_ctr_complexity=4, one_hot_max_size=2)

def clean(df):
    """Rename columns to LightGBM-safe names."""
    return df.rename(columns=lambda c: re.sub(r'[^0-9a-zA-Z_]', '_', c))

mlp_fill = X.median()

print(f"X: {X.shape} | X_test: {X_test.shape} | y mean: {y.mean():.4f} | folds: {N_FOLDS} | device: {DEVICE}")

In [ ]:
import time
from huggingface_hub import HfApi, create_repo, snapshot_download

HF_REPO    = "sujalgawas/airline-satisfaction-ckpt"
HF_PRIVATE = True

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

hf_api = HfApi(token=HF_TOKEN)
HF_OK = False
try:
    create_repo(HF_REPO, repo_type="model", private=HF_PRIVATE, exist_ok=True, token=HF_TOKEN)
    snapshot_download(repo_id=HF_REPO, repo_type="model", local_dir=CKPT_DIR, token=HF_TOKEN)
    HF_OK = True
    restored = [f for f in os.listdir(CKPT_DIR) if f.endswith(('.pkl', '.json'))]
    print(f"HF repo ready: {HF_REPO} | restored {len(restored)} checkpoint files into {CKPT_DIR}/")
except Exception as e:
    print(f"HF setup failed -> checkpoints stay LOCAL only ({e})")

def hf_push(path, retries=3):
    """Upload one checkpoint file to the Hub. Never raises, so a network hiccup can't kill training."""
    if not HF_OK or not os.path.exists(path):
        return
    name = os.path.basename(path)
    for a in range(retries):
        try:
            hf_api.upload_file(
                path_or_fileobj=path, path_in_repo=name,
                repo_id=HF_REPO, repo_type="model",
                commit_message=f"ckpt {name}",
            )
            print(f"  pushed {name} -> HF")
            return
        except Exception as e:
            print(f"  HF push failed for {name} ({a + 1}/{retries}): {e}")
            time.sleep(5 * (a + 1))


In [ ]:
X_extra = y_extra = None
need_extra = any(n.startswith('og_') for n in MODELS_ON)

if need_extra and not EXTRA_DATA_PATH:
    print("EXTRA_DATA_PATH is None -> skipping the og_* models")
    MODELS_ON = [n for n in MODELS_ON if not n.startswith('og_')]
elif need_extra:
    paths = [EXTRA_DATA_PATH] if isinstance(EXTRA_DATA_PATH, str) else list(EXTRA_DATA_PATH)
    ex = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
    missing = [c for c in raw_features if c not in ex.columns]
    if missing:
        print(f"Extra data missing columns {missing} -> skipping og_* models")
        MODELS_ON = [n for n in MODELS_ON if not n.startswith('og_')]
    else:
        y_extra = (
            ex['satisfaction']
            .map(lambda v: int(str(v).strip().lower() in ('satisfied', 'true', '1')))
            .reset_index(drop=True)
        )
        # Flag external rows with is_orig=1 so trees can distinguish original vs synthetic distributions
        ex_fe = add_features(encode_categoricals(ex[raw_features]), is_orig=1).reset_index(drop=True)
        # IMPORTANT FIX: Compute leak-free OOF target encoding inside X_extra itself (never using validation fold targets!)
        ex_folds = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(ex_fe, y_extra))
        ex_te, _ = compute_oof_target_encoding(ex_fe, ex_fe.iloc[:10], y_extra, ex_folds, m_smooth=20.0)
        X_extra = pd.concat([ex_fe, ex_te], axis=1)[features].copy()
        X_extra[cat_feature_names] = X_extra[cat_feature_names].astype('int64')
        print(f"Extra training rows loaded (leak-free): {len(X_extra)} | pos rate {y_extra.mean():.3f}")

def with_extra(Xd, tr, extra, return_weights=False):
    xt, yt = Xd.iloc[tr], y.iloc[tr]
    if extra is None:
        if return_weights:
            return xt, yt, np.ones(len(xt), dtype=np.float32)
        return xt, yt
    xt_comb = pd.concat([xt, extra], ignore_index=True)
    yt_comb = pd.concat([yt, y_extra], ignore_index=True)
    if return_weights:
        w = np.concatenate([
            np.ones(len(xt), dtype=np.float32),
            np.full(len(extra), EXTRA_WEIGHT, dtype=np.float32)
        ])
        return xt_comb, yt_comb, w
    return xt_comb, yt_comb

def view(name, df):
    """Return data in the exact format each model expects."""
    if name in ('lgb', 'og_lgb'):
        return clean(df)
    if name in ('cat', 'og_cat'):
        out = df.copy()
        out[cat_feature_names] = out[cat_feature_names].astype('int64')
        return out
    if name == 'mlp':
        out = df.fillna(mlp_fill).copy()
        for c in mlp_cat_cols:
            out[c] = out[c].astype('int64').astype('category')
        return out
    return df

def ckpt_path(name, k, cfg):
    key = hashlib.md5(repr((name, cfg, N_FOLDS, tuple(features), len(X),
                            len(X_extra) if (name.startswith('og_') and X_extra is not None) else None
                            )).encode()).hexdigest()[:10]
    return f'{CKPT_DIR}/{name}_f{k}_{key}.pkl'

def load_ckpt(name, k, cfg):
    p = ckpt_path(name, k, cfg)
    if USE_CKPT and os.path.exists(p):
        try:
            return joblib.load(p)
        except Exception as e:
            print(f"  could not load {p} ({e}); retraining")
    return None

def save_ckpt(m, name, k, cfg):
    if USE_CKPT:
        p = ckpt_path(name, k, cfg)
        try:
            joblib.dump(m, p)
        except Exception as e:
            print(f"  could not save {name} fold {k} ({e})")
            return
        hf_push(p) 

fold_models = {}

### 3.1 XGBoost

In [ ]:
if 'xgb' in MODELS_ON:
    xgb_file = lambda k: f'{CKPT_DIR}/xgb_{N_FOLDS}f_fold{k}.json'
    have_disk = all(os.path.exists(xgb_file(k)) for k in range(N_FOLDS))

    if not TRAIN_XGB and not have_disk:
        TRAIN_XGB = True

    def previous_booster(k):
        if RESUME and os.path.exists(xgb_file(k)):
            b = xgboost.Booster()
            b.load_model(xgb_file(k))
            return b
        return None

    if TRAIN_XGB:
        xgb_models = []
        for k, (tr, va) in enumerate(folds):
            if (not RESUME) and os.path.exists(xgb_file(k)):
                m = XGBClassifier()
                m.load_model(xgb_file(k))
                m.set_params(device='cpu')
                xgb_models.append(m)
                print(f"xgb fold {k}: loaded from checkpoint, skipping training")
                continue
            prev = previous_booster(k)
            lr_, trees_, pat_ = ((XGB_LR, XGB_NEW_TREES, XGB_PATIENCE)
                                 if prev is not None else XGB_SCRATCH)
            print(f"xgb fold {k}: {'continuing' if prev is not None else 'from scratch'} (lr {lr_})")
            m = XGBClassifier(
                n_estimators=trees_, learning_rate=lr_, **xgb_params,
                tree_method="hist", device=DEVICE, eval_metric="auc",
                early_stopping_rounds=pat_, random_state=42 + k,
            )
            m.fit(X.iloc[tr], y.iloc[tr],
                  eval_set=[(X.iloc[va], y.iloc[va])],
                  verbose=2000, xgb_model=prev)
            m.save_model(xgb_file(k))
            hf_push(xgb_file(k))
            m.set_params(device='cpu')
            xgb_models.append(m)
            print(f"xgb fold {k}: val AUC {m.best_score:.5f}")
        fold_models['xgb'] = xgb_models
    else:
        xgb_models = []
        for k in range(N_FOLDS):
            m = XGBClassifier()
            m.load_model(xgb_file(k))
            m.set_params(device='cpu')
            xgb_models.append(m)
        fold_models['xgb'] = xgb_models

### 3.2 LightGBM (`lgb` & Leak-Free Weighted `og_lgb`)

In [ ]:
lgb_cat_cols = [re.sub(r'[^0-9a-zA-Z_]', '_', c) for c in mlp_cat_cols]

def train_lgb(name, extra):
    Xl = clean(X)
    Xl_extra = clean(extra) if extra is not None else None
    cfg = (lgb_params, LGB_LR, LGB_TREES, LGB_PATIENCE, EXTRA_WEIGHT if extra is not None else 1.0)
    out = []
    for k, (tr, va) in enumerate(folds):
        m = load_ckpt(name, k, cfg)
        if m is None:
            m = lgb.LGBMClassifier(
                n_estimators=LGB_TREES, learning_rate=LGB_LR,
                random_state=42 + k, verbose=-1, n_jobs=-1, **lgb_params,
            )
            Xt, yt, sw = with_extra(Xl, tr, Xl_extra, return_weights=True)
            m.fit(
                Xt, yt,
                sample_weight=sw,
                eval_set=[(Xl.iloc[va], y.iloc[va])],
                eval_metric='auc',
                categorical_feature=lgb_cat_cols,
                callbacks=[lgb.early_stopping(LGB_PATIENCE, verbose=False),
                           lgb.log_evaluation(1000)]
            )
            save_ckpt(m, name, k, cfg)
        out.append(m)
        print(f"{name} fold {k}: val AUC {m.best_score_['valid_0']['auc']:.5f} | trees {m.best_iteration_}")
    return out

if 'lgb' in MODELS_ON:
    fold_models['lgb'] = train_lgb('lgb', None)
if 'og_lgb' in MODELS_ON:
    fold_models['og_lgb'] = train_lgb('og_lgb', X_extra)

### 3.3 CatBoost (`cat` & Leak-Free Weighted `og_cat`)

In [ ]:
from catboost import Pool

def train_cat(name, extra):
    cfg = (cat_params, CAT_LR, CAT_TREES, CAT_PATIENCE, EXTRA_WEIGHT if extra is not None else 1.0)
    out = []
    for k, (tr, va) in enumerate(folds):
        m = load_ckpt(name, k, cfg)
        if m is None:
            m = CatBoostClassifier(
                iterations=CAT_TREES, learning_rate=CAT_LR,
                early_stopping_rounds=CAT_PATIENCE,
                task_type='GPU' if HAS_GPU else 'CPU',
                random_seed=42 + k, verbose=1000, **cat_params,
            )
            Xt, yt, sw = with_extra(X, tr, extra, return_weights=True)
            Xt = Xt.copy()
            Xt[cat_feature_names] = Xt[cat_feature_names].astype('int64')
            tr_pool = Pool(Xt, yt, cat_features=cat_feature_names, weight=sw)
            va_pool = Pool(X.iloc[va], y.iloc[va], cat_features=cat_feature_names)
            m.fit(tr_pool, eval_set=va_pool)
            save_ckpt(m, name, k, cfg)
        out.append(m)
        print(f"{name} fold {k}: val AUC {m.get_best_score()['validation']['AUC']:.5f} | trees {m.get_best_iteration()}")
    return out

if 'cat' in MODELS_ON:
    fold_models['cat'] = train_cat('cat', None)
if 'og_cat' in MODELS_ON:
    fold_models['og_cat'] = train_cat('og_cat', X_extra)

### 3.4 Fast GPU PyTorch Tabular ResNet (`resnet`)

A lightweight, fast (~30 sec/fold on GPU) **Tabular Residual Network with Entity Embeddings, SiLU + BatchNorm + Dropout Residual Blocks, and OneCycleLR / Cosine Annealing**.
- Provides genuine neural network probability diversity alongside GBDTs without any PyTorch Lightning multi-GPU subprocess issues.
- Wraps cleanly in a scikit-learn compatible `predict_proba` interface.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import RobustScaler, QuantileTransformer

class _TabResNetModule(nn.Module):
    def __init__(self, cat_dims, n_num, hidden_dim=256, n_blocks=3, dropout=0.20):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(num_classes, emb_dim) for num_classes, emb_dim in cat_dims
        ])
        emb_total = sum(emb_dim for _, emb_dim in cat_dims)
        in_dim = emb_total + n_num

        self.in_proj = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.SiLU(),
        )
        blocks = []
        for _ in range(n_blocks):
            blocks.append(nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.SiLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
            ))
        self.blocks = nn.ModuleList(blocks)
        self.act = nn.SiLU()
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, x_cat, x_num):
        embs = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
        x = torch.cat(embs + [x_num], dim=1)
        h = self.in_proj(x)
        for blk in self.blocks:
            h = self.act(h + blk(h))
        return self.head(h).squeeze(-1)


class TabularResNetClassifier:
    """Self-contained GPU Tabular ResNet with entity embeddings and early stopping on ROC AUC."""
    def __init__(self, cat_cols, epochs=22, batch_size=2048, lr=2e-3, device='cuda:0', random_state=42):
        self.cat_cols = list(cat_cols)
        self.epochs = epochs
        self.batch_size = batch_size
        self.lr = lr
        self.device = device
        self.random_state = random_state

    def _encode_Preprocessor(self, df, fit=False):
        if fit:
            self.num_cols_ = [c for c in df.columns if c not in self.cat_cols]
            self.cat_maps_ = {}
            self.cat_dims_ = []
            for c in self.cat_cols:
                uniq = sorted(df[c].dropna().unique())
                mp = {v: i + 1 for i, v in enumerate(uniq)}  # 0 reserved for unknown
                self.cat_maps_[c] = mp
                n_vocab = len(uniq) + 2
                emb_dim = min(32, max(4, (n_vocab + 1) // 2))
                self.cat_dims_.append((n_vocab, emb_dim))
            self.num_medians_ = df[self.num_cols_].median()
            self.scaler_ = RobustScaler()
            num_arr = self.scaler_.fit_transform(df[self.num_cols_].fillna(self.num_medians_).values.astype(np.float32))
        else:
            num_arr = self.scaler_.transform(df[self.num_cols_].fillna(self.num_medians_).values.astype(np.float32))

        num_arr = np.clip(num_arr, -10.0, 10.0)
        cat_arr = np.zeros((len(df), len(self.cat_cols)), dtype=np.int64)
        for i, c in enumerate(self.cat_cols):
            cat_arr[:, i] = df[c].map(self.cat_maps_[c]).fillna(0).astype(np.int64).values
        return cat_arr, num_arr

    def fit(self, X_tr, y_tr, X_va, y_va):
        torch.manual_seed(self.random_state)
        np.random.seed(self.random_state)

        xc_tr, xn_tr = self._encode_Preprocessor(X_tr, fit=True)
        xc_va, xn_va = self._encode_Preprocessor(X_va, fit=False)

        ds_tr = TensorDataset(torch.from_numpy(xc_tr), torch.from_numpy(xn_tr), torch.from_numpy(y_tr.values.astype(np.float32)))
        dl_tr = DataLoader(ds_tr, batch_size=self.batch_size, shuffle=True, drop_last=False)

        model = _TabResNetModule(self.cat_dims_, xn_tr.shape[1]).to(self.device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=self.lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.epochs, eta_min=1e-5)
        criterion = nn.BCEWithLogitsLoss()

        best_auc = -1.0
        best_state = None

        for epoch in range(self.epochs):
            model.train()
            for bc, bn, by in dl_tr:
                bc, bn, by = bc.to(self.device), bn.to(self.device), by.to(self.device)
                optimizer.zero_grad()
                logits = model(bc, bn)
                loss = criterion(logits, by)
                loss.backward()
                optimizer.step()
            scheduler.step()

            val_p = self._predict_raw(model, xc_va, xn_va)
            auc = roc_auc_score(y_va, val_p)
            if auc > best_auc:
                best_auc = auc
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        model.load_state_dict(best_state)
        self.model_ = model.cpu().eval()
        self.best_score_ = best_auc
        return self

    def _predict_raw(self, model, xc, xn):
        model.eval()
        preds = []
        dev = next(model.parameters()).device
        with torch.no_grad():
            for i in range(0, len(xc), self.batch_size * 2):
                bc = torch.from_numpy(xc[i:i + self.batch_size * 2]).to(dev)
                bn = torch.from_numpy(xn[i:i + self.batch_size * 2]).to(dev)
                p = torch.sigmoid(model(bc, bn)).cpu().numpy()
                preds.append(p)
        return np.concatenate(preds)

    def predict_proba(self, df):
        xc, xn = self._encode_Preprocessor(df, fit=False)
        self.model_.to(self.device)
        p1 = self._predict_raw(self.model_, xc, xn)
        self.model_.cpu()
        return np.column_stack([1.0 - p1, p1])

if 'resnet' in MODELS_ON:
    cfg = ('resnet', RESNET_EPOCHS, RESNET_BATCH, tuple(mlp_cat_cols))
    resnet_models = []
    for k, (tr, va) in enumerate(folds):
        m = load_ckpt('resnet', k, cfg)
        if m is None:
            m = TabularResNetClassifier(
                cat_cols=mlp_cat_cols, epochs=RESNET_EPOCHS,
                batch_size=RESNET_BATCH, device=DEVICE, random_state=42 + k
            )
            m.fit(X.iloc[tr], y.iloc[tr], X.iloc[va], y.iloc[va])
            save_ckpt(m, 'resnet', k, cfg)
        resnet_models.append(m)
        print(f"resnet fold {k}: val AUC {m.best_score_:.5f}")
    fold_models['resnet'] = resnet_models

### 3.5 RealMLP (Pinned to Single GPU so DDP Never Hangs in Jupyter)

In v2, `RealMLP_TD_Classifier` saw `CUDA_VISIBLE_DEVICES: [0,1]` on Kaggle Dual-T4 and attempted to launch multi-GPU distributed training inside an interactive notebook—hanging for 42 minutes before failing with `SystemExit: 1`.
- Below we temporarily restrict `CUDA_VISIBLE_DEVICES="0"` and pass `device='cuda:0'` with `n_epochs=25, batch_size=2048`, plus a clean `try/except` fallback so a single slow fold can never crash the notebook.

In [ ]:
if 'mlp' in MODELS_ON:
    Xm = view('mlp', X)
    cfg = (MLP_EPOCHS, MLP_BATCH, 'cuda:0' if HAS_GPU else 'cpu', tuple(mlp_cat_cols))
    mlp_models = []
    prev_cuda_vis = os.environ.get('CUDA_VISIBLE_DEVICES', None)
    if HAS_GPU:
        os.environ['CUDA_VISIBLE_DEVICES'] = '0'
    try:
        for k, (tr, va) in enumerate(folds):
            m = load_ckpt('mlp', k, cfg)
            if m is None:
                kw = dict(
                    device='cuda:0' if HAS_GPU else 'cpu',
                    random_state=42 + k,
                    n_epochs=MLP_EPOCHS,
                    batch_size=MLP_BATCH,
                    val_metric_name='1-auc_ovr',
                    verbosity=0,
                    cat_col_names=mlp_cat_cols
                )
                try:
                    m = RealMLP_TD_Classifier(**kw)
                except TypeError:
                    m = RealMLP_TD_Classifier(device='cuda:0' if HAS_GPU else 'cpu', random_state=42 + k)
                m.fit(Xm.iloc[tr], y.iloc[tr], Xm.iloc[va], y.iloc[va])
                save_ckpt(m, 'mlp', k, cfg)
            mlp_models.append(m)
            val_p = m.predict_proba(Xm.iloc[va])[:, 1]
            print(f"mlp fold {k}: val AUC {roc_auc_score(y.iloc[va], val_p):.5f}")
        fold_models['mlp'] = mlp_models
    except (Exception, KeyboardInterrupt, SystemExit) as e:
        print(f"RealMLP interrupted or skipped ({e}); continuing with {list(fold_models.keys())}")
        MODELS_ON = [n for n in MODELS_ON if n != 'mlp']
    finally:
        if prev_cuda_vis is not None:
            os.environ['CUDA_VISIBLE_DEVICES'] = prev_cuda_vis
        elif 'CUDA_VISIBLE_DEVICES' in os.environ:
            del os.environ['CUDA_VISIBLE_DEVICES']

In [ ]:
from scipy.stats import rankdata
from scipy.special import expit, logit
from scipy.optimize import minimize

names = [n for n in MODELS_ON if n in fold_models]
eps = 1e-6

def to_logit(p):
    return logit(np.clip(p, eps, 1.0 - eps))

def to_rank(o):
    return rankdata(o) / (len(o) + 1.0)

def fold_oof(name):
    out = np.zeros(len(X))
    for (tr, va), m in zip(folds, fold_models[name]):
        out[va] = m.predict_proba(view(name, X.iloc[va]))[:, 1]
    return out

oofs = {n: fold_oof(n) for n in names}
for n, o in oofs.items():
    acc_05 = accuracy_score(y, (o >= 0.5).astype(int))
    print(f"{n:7s} OOF AUC: {roc_auc_score(y, o):.5f} | LogLoss: {log_loss(y, o):.5f} | Acc@0.5: {acc_05:.5f}")

def hill_climb_auc(preds, n_iter=300):
    """Fine-grained greedy ensemble selection for ROC AUC."""
    picks, total, best = [], np.zeros(len(y)), (-1.0, 0)
    for i in range(n_iter):
        cand = {n: roc_auc_score(y, (total + o) / (i + 1)) for n, o in preds.items()}
        n = max(cand, key=cand.get)
        picks.append(n)
        total = total + preds[n]
        if cand[n] > best[0]:
            best = (cand[n], i + 1)
    picks = picks[:best[1]]
    return {n: round(picks.count(n) / len(picks), 4) for n in preds}, best[0]

# 1. Optimize Logit-Space Weights (Hill-Climb + Nelder-Mead refinement)
logit_dict = {n: to_logit(o) for n in names for o in [oofs[n]]}
w_logit_hc, auc_logit_hc = hill_climb_auc(logit_dict, n_iter=300)

L_mat = np.column_stack([logit_dict[n] for n in names])
w0 = np.array([w_logit_hc[n] for n in names], dtype=np.float64)

# Refine logit weights via SLSQP minimizing LogLoss + maximizing smooth AUC surrogate
def _loss_fn(w):
    w = np.maximum(0, w)
    if w.sum() == 0:
        return 1e9
    w = w / w.sum()
    p = expit(L_mat @ w)
    return log_loss(y_arr, p)

res_ll = minimize(_loss_fn, w0, method='SLSQP', bounds=[(0, 1)] * len(names),
                  constraints={'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0})
w_slsqp = res_ll.x / np.sum(res_ll.x)
auc_slsqp = roc_auc_score(y_arr, expit(L_mat @ w_slsqp))

if auc_slsqp > auc_logit_hc:
    w_logit = {n: round(float(w_slsqp[i]), 4) for i, n in enumerate(names)}
    auc_logit = auc_slsqp
else:
    w_logit = w_logit_hc
    auc_logit = auc_logit_hc

# 2. Direct Coordinate-Ascent Optimization for OOF Accuracy in Logit Space
def optimize_accuracy_logit_weights(L_matrix, y_true, w_init, n_passes=3):
    w = np.array([w_init[n] for n in names], dtype=np.float64)
    w = w / w.sum()
    thr_candidates = np.linspace(-0.35, 0.35, 141)

    def eval_w(weights):
        comb_l = L_matrix @ weights
        accs = [np.mean((comb_l >= t) == y_true) for t in thr_candidates]
        best_idx = int(np.argmax(accs))
        return accs[best_idx], thr_candidates[best_idx]

    best_acc, best_l_thr = eval_w(w)
    for _ in range(n_passes):
        for i in range(len(names)):
            for delta in (-0.08, -0.04, -0.02, -0.01, 0.01, 0.02, 0.04, 0.08):
                w_try = w.copy()
                w_try[i] = max(0.0, w_try[i] + delta)
                if w_try.sum() <= 0:
                    continue
                w_try /= w_try.sum()
                acc_try, thr_try = eval_w(w_try)
                if acc_try > best_acc:
                    best_acc, best_l_thr, w = acc_try, thr_try, w_try
    return {n: round(float(w[i]), 4) for i, n in enumerate(names)}, best_acc, float(expit(best_l_thr))

w_acc, best_acc_blend, best_thr_acc = optimize_accuracy_logit_weights(L_mat, y_arr, w_logit)

print(f"\nCalibrated Logit Blend (AUC-opt) | weights: {w_logit} | OOF AUC: {auc_logit:.5f}")
print(f"Direct Accuracy Blend  (Acc-opt) | weights: {w_acc} | OOF Acc: {best_acc_blend:.5f} @ thr {best_thr_acc:.4f}")

In [ ]:
def apply_logit_blend(P_dict, weight_map=None):
    """Always returns well-calibrated probabilities in (0, 1) — fixes the v2 rank-to-logit distortion bug."""
    w_use = weight_map if weight_map is not None else w_logit
    return expit(sum(w_use[n] * to_logit(P_dict[n]) for n in names))

oof_blend     = apply_logit_blend(oofs, w_logit)
oof_acc_blend = apply_logit_blend(oofs, w_acc)

def make_meta(P, data):
    """Build rich calibrated meta-features for Level-2 stacking judges."""
    probs = np.column_stack([np.clip(P[n], eps, 1.0 - eps) for n in names])
    logits = np.log(probs / (1.0 - probs))
    M = pd.DataFrame({f'logit_{n}': logits[:, i] for i, n in enumerate(names)})

    for i, n in enumerate(names):
        M[f'conf_{n}'] = np.abs(probs[:, i] - 0.5) * 2.0

    # Pairwise model disagreement features (helps meta-learner arbitrate when GBDTs & NNs disagree)
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            M[f'diff_{names[i]}_{names[j]}'] = logits[:, i] - logits[:, j]

    p_b = apply_logit_blend(P, w_logit)
    M['logit_blend']     = to_logit(p_b)
    M['logit_acc_blend'] = to_logit(apply_logit_blend(P, w_acc))
    M['logit_mean']      = logits.mean(1)
    M['logit_std']       = logits.std(1)
    M['logit_median']    = np.median(logits, axis=1)
    M['logit_range']     = logits.max(1) - logits.min(1)
    M['p_mean']          = probs.mean(1)
    M['p_std']           = probs.std(1)
    M['binary_entropy']  = -(p_b * np.log(np.clip(p_b, eps, 1.0)) + (1.0 - p_b) * np.log(np.clip(1.0 - p_b, eps, 1.0)))
    M['n_votes']         = (probs >= 0.5).sum(1)
    raw = clean(data).reset_index(drop=True)
    return pd.concat([M, raw], axis=1)

logit_cols = [f'logit_{n}' for n in names] + ['logit_blend', 'logit_acc_blend', 'logit_std', 'binary_entropy']
M_train = make_meta(oofs, X)

oof_lr, oof_gbm, oof_cat_meta = np.zeros(len(X)), np.zeros(len(X)), np.zeros(len(X))
lr_models, gbm_models, cat_meta_models = [], [], []

for k, (tr, va) in enumerate(folds):
    # 1. L2-Regularized Logistic Regression Judge
    lr = LogisticRegression(C=0.25, max_iter=1000)
    lr.fit(M_train.iloc[tr][logit_cols], y_arr[tr])
    oof_lr[va] = lr.predict_proba(M_train.iloc[va][logit_cols])[:, 1]
    lr_models.append(lr)

    # 2. Shallow Calibrated LightGBM Judge
    g = lgb.LGBMClassifier(
        n_estimators=STACK_ROUNDS, learning_rate=0.012, num_leaves=15,
        min_child_samples=300, subsample=0.8, subsample_freq=1,
        colsample_bytree=0.65, reg_lambda=15.0, reg_alpha=3.0,
        random_state=42 + k, verbose=-1, n_jobs=-1,
    )
    g.fit(
        M_train.iloc[tr], y_arr[tr],
        eval_set=[(M_train.iloc[va], y_arr[va])],
        eval_metric='auc',
        callbacks=[lgb.early_stopping(STACK_PATIENCE, verbose=False), lgb.log_evaluation(0)]
    )
    va_raw_prob = g.predict_proba(M_train.iloc[va])[:, 1]
    va_logit = to_logit(va_raw_prob)
    ref_logit = M_train.iloc[va]['logit_blend'].values
    va_logit_cal = (va_logit - va_logit.mean()) / (va_logit.std() + 1e-6) * ref_logit.std() + ref_logit.mean()
    oof_gbm[va] = expit(va_logit_cal)
    gbm_models.append(g)

    # 3. Shallow Depth-4 CatBoost Meta-Judge
    cb_meta = CatBoostClassifier(
        iterations=2500, learning_rate=0.02, depth=4, l2_leaf_reg=15.0,
        eval_metric='AUC', early_stopping_rounds=200,
        task_type='GPU' if HAS_GPU else 'CPU', random_seed=42 + k, verbose=0
    )
    cb_meta.fit(M_train.iloc[tr][logit_cols], y_arr[tr], eval_set=(M_train.iloc[va][logit_cols], y_arr[va]))
    cb_raw_prob = cb_meta.predict_proba(M_train.iloc[va][logit_cols])[:, 1]
    cb_logit = to_logit(cb_raw_prob)
    cb_logit_cal = (cb_logit - cb_logit.mean()) / (cb_logit.std() + 1e-6) * ref_logit.std() + ref_logit.mean()
    oof_cat_meta[va] = expit(cb_logit_cal)
    cat_meta_models.append(cb_meta)

    print(f"stack fold {k}: LGBM-judge AUC {g.best_score_['valid_0']['auc']:.5f} ({g.best_iteration_} trees) | Cat-judge AUC {cb_meta.get_best_score()['validation']['AUC']:.5f}")

In [ ]:
meta_cands = {
    'blend': oof_blend,
    'acc_blend': oof_acc_blend,
    'lr': oof_lr,
    'gbm': oof_gbm,
    'cat_meta': oof_cat_meta,
}

# 1. Super-Ensemble optimized for ROC AUC
super_weights, super_auc = hill_climb_auc({k: to_logit(v) for k, v in meta_cands.items()}, n_iter=300)
oof_super = expit(sum(super_weights[k] * to_logit(meta_cands[k]) for k in meta_cands))

# 2. Super-Ensemble directly optimized for OOF Classification Accuracy
meta_L_mat = np.column_stack([to_logit(meta_cands[k]) for k in meta_cands])
meta_names = list(meta_cands.keys())
w_super_acc_arr = np.array([super_weights[k] for k in meta_names], dtype=np.float64)
w_super_acc_arr /= w_super_acc_arr.sum()

thr_fine_grid = np.linspace(0.35, 0.65, 601)
for _ in range(3):
    for i in range(len(meta_names)):
        for delta in (-0.05, -0.02, -0.01, 0.01, 0.02, 0.05):
            w_t = w_super_acc_arr.copy()
            w_t[i] = max(0.0, w_t[i] + delta)
            if w_t.sum() <= 0:
                continue
            w_t /= w_t.sum()
            p_t = expit(meta_L_mat @ w_t)
            acc_t = max(accuracy_score(y_arr, (p_t >= th).astype(int)) for th in np.linspace(0.45, 0.55, 41))
            cur_p = expit(meta_L_mat @ w_super_acc_arr)
            cur_acc = max(accuracy_score(y_arr, (cur_p >= th).astype(int)) for th in np.linspace(0.45, 0.55, 41))
            if acc_t > cur_acc:
                w_super_acc_arr = w_t

super_acc_weights = {meta_names[i]: round(float(w_super_acc_arr[i]), 4) for i in range(len(meta_names))}
oof_super_acc = expit(sum(super_acc_weights[k] * to_logit(meta_cands[k]) for k in meta_cands))

cands = {
    **meta_cands,
    'super_auc': oof_super,
    'super_acc': oof_super_acc,
}

thr_of, acc_of, acc05_of = {}, {}, {}
for c, o in cands.items():
    t = max(thr_fine_grid, key=lambda t: SCORE_FN(y_arr, (o >= t).astype(int)))
    thr_of[c] = float(t)
    acc_of[c] = accuracy_score(y_arr, (o >= t).astype(int))
    acc05_of[c] = accuracy_score(y_arr, (o >= 0.50).astype(int))
    print(f"{c:10s} | OOF AUC: {roc_auc_score(y_arr, o):.5f} | Max Acc: {acc_of[c]:.5f} @ thr {t:.4f} | Acc@0.50: {acc05_of[c]:.5f}")

# Select best candidate depending on whether we prioritize probability ranking (AUC) or strict Accuracy
if SUBMISSION_TYPE in ('binary', 'calibrated_proba'):
    STACK_CHOICE = max(cands, key=lambda c: acc_of[c])
else:
    STACK_CHOICE = max(cands, key=lambda c: roc_auc_score(y_arr, cands[c]))

print(f"\n-> Selected strategy: {STACK_CHOICE}")
print(f"   super_auc weights: {super_weights}")
print(f"   super_acc weights: {super_acc_weights}")

def base_test_preds(data):
    """Average fold probabilities for each base model."""
    return {
        n: np.mean([m.predict_proba(view(n, data))[:, 1] for m in fold_models[n]], axis=0)
        for n in names
    }

def predict_proba_avg(data):
    """Final test prediction using the winning strategy."""
    P = base_test_preds(data)
    p_blend     = apply_logit_blend(P, w_logit)
    p_acc_blend = apply_logit_blend(P, w_acc)
    if STACK_CHOICE == 'blend':
        return p_blend
    if STACK_CHOICE == 'acc_blend':
        return p_acc_blend

    M = make_meta(P, data)
    p_lr = np.mean([m.predict_proba(M[logit_cols])[:, 1] for m in lr_models], axis=0)
    if STACK_CHOICE == 'lr':
        return p_lr

    ref_logit = to_logit(p_blend)
    gbm_fold_logits = []
    for m in gbm_models:
        fl = to_logit(m.predict_proba(M)[:, 1])
        fl_cal = (fl - fl.mean()) / (fl.std() + 1e-6) * ref_logit.std() + ref_logit.mean()
        gbm_fold_logits.append(fl_cal)
    p_gbm = expit(np.mean(gbm_fold_logits, axis=0))
    if STACK_CHOICE == 'gbm':
        return p_gbm

    cat_fold_logits = []
    for m in cat_meta_models:
        fl = to_logit(m.predict_proba(M[logit_cols])[:, 1])
        fl_cal = (fl - fl.mean()) / (fl.std() + 1e-6) * ref_logit.std() + ref_logit.mean()
        cat_fold_logits.append(fl_cal)
    p_cat_meta = expit(np.mean(cat_fold_logits, axis=0))
    if STACK_CHOICE == 'cat_meta':
        return p_cat_meta

    meta_test = {
        'blend': p_blend,
        'acc_blend': p_acc_blend,
        'lr': p_lr,
        'gbm': p_gbm,
        'cat_meta': p_cat_meta,
    }
    w_final = super_acc_weights if STACK_CHOICE == 'super_acc' else super_weights
    return expit(sum(w_final[k] * to_logit(meta_test[k]) for k in meta_test))

oof = cands[STACK_CHOICE]
best_thr = thr_of[STACK_CHOICE]
y_pred = (oof >= best_thr).astype(int)

print("\nBest threshold (OOF):", round(best_thr, 4))
print("AUC:      ", roc_auc_score(y, oof))
print("Accuracy: ", accuracy_score(y, y_pred))
print("Precision:", precision_score(y, y_pred))
print("Recall:   ", recall_score(y, y_pred))
print("F1:       ", f1_score(y, y_pred))

## 6. Submission Generation

Supports three modes via `SUBMISSION_TYPE`:
- `'proba'` (default): Outputs continuous, well-calibrated probabilities in `[0, 1]` for ROC AUC evaluation.
- `'calibrated_proba'`: Shifts test log-odds by `-logit(best_thr)` so that any downstream evaluator applying a standard `0.50` cutoff automatically uses the optimal OOF accuracy threshold (`best_thr`) while retaining continuous probabilities!
- `'binary'`: Outputs hard `0` / `1` integer predictions at `best_thr`.

In [ ]:
test_proba = predict_proba_avg(X_test)

if SUBMISSION_TYPE == 'binary':
    sub_vals = (test_proba >= best_thr).astype(int)
elif SUBMISSION_TYPE == 'calibrated_proba':
    # Shift logits so that p == best_thr maps to exactly 0.50
    sub_vals = expit(to_logit(test_proba) - to_logit(best_thr))
else:
    sub_vals = np.clip(test_proba, 0.0, 1.0)

output = pd.DataFrame({
    'id': test_pd['id'],
    'satisfaction': sub_vals
})

output.to_csv('submission.csv', index=False)
print(f"Saved submission.csv ({SUBMISSION_TYPE} mode) with {len(output)} rows")

# Also save threshold-calibrated & hard-label files for convenience
pd.DataFrame({
    'id': test_pd['id'],
    'satisfaction': expit(to_logit(test_proba) - to_logit(best_thr))
}).to_csv('submission_calibrated_thr05.csv', index=False)

pd.DataFrame({
    'id': test_pd['id'],
    'satisfaction': (test_proba >= best_thr).astype(int)
}).to_csv('submission_binary_acc.csv', index=False)

output.head(10)

In [ ]:
# Safety net: sync the whole checkpoint folder to HF (only uploads what's missing/changed)
if HF_OK:
    hf_api.upload_folder(
        folder_path=CKPT_DIR, repo_id=HF_REPO, repo_type="model",
        allow_patterns=["*.pkl", "*.json"], commit_message="sync all checkpoints",
    )
    print("All checkpoints synced to", HF_REPO)
